In [1]:
pip install pymongo

Note: you may need to restart the kernel to use updated packages.


In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
import pymongo
from pymongo import MongoClient

In [4]:
from pymongo import MongoClient

mongo = MongoClient("mongodb://mongodb_db:27017/")

In [5]:
from pymongo import MongoClient

mongo = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

print(mongo.admin.command("ping"))

{'ok': 1.0}


In [6]:
db = mongo["test"]
list(db.students.find())

[]

In [7]:
students = [
    {"id": 1, "name": "Shahd", "age": 20, "department": "AI", "grade": 90},
    {"id": 2, "name": "Hassan", "age": 21, "department": "CS", "grade": 85},
    {"id": 3, "name": "Anas", "age": 20, "department": "AI", "grade": 95},
    {"id": 4, "name": "Mariam", "age": 22, "department": "CS", "grade": 78},
    {"id": 5, "name": "Omar", "age": 21, "department": "IS", "grade": 88}
]

print(students)

[{'id': 1, 'name': 'Shahd', 'age': 20, 'department': 'AI', 'grade': 90}, {'id': 2, 'name': 'Hassan', 'age': 21, 'department': 'CS', 'grade': 85}, {'id': 3, 'name': 'Anas', 'age': 20, 'department': 'AI', 'grade': 95}, {'id': 4, 'name': 'Mariam', 'age': 22, 'department': 'CS', 'grade': 78}, {'id': 5, 'name': 'Omar', 'age': 21, 'department': 'IS', 'grade': 88}]


In [8]:
from pymongo import MongoClient

mongo = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)

print(mongo.admin.command("ping"))

db = mongo["students_db"]
collection = db["students"]

if collection.count_documents({}) == 0:
    collection.insert_many(students)

print("Number of students:", collection.count_documents({}))
print(list(collection.find({}, {"_id": 0})))

{'ok': 1.0}
Number of students: 5
[{'id': 1, 'name': 'Shahd', 'age': 20, 'department': 'AI', 'grade': 90}, {'id': 2, 'name': 'Hassan', 'age': 21, 'department': 'CS', 'grade': 85}, {'id': 3, 'name': 'Anas', 'age': 20, 'department': 'AI', 'grade': 95}, {'id': 4, 'name': 'Mariam', 'age': 22, 'department': 'CS', 'grade': 78}, {'id': 5, 'name': 'Omar', 'age': 21, 'department': 'IS', 'grade': 88}]


In [9]:
db = mongo["students_db"]
collection = db["students"]

if collection.count_documents({}) == 0:
    collection.insert_many(students)

print("Number of students:", collection.count_documents({}))
print(list(collection.find({}, {"_id": 0})))

Number of students: 5
[{'id': 1, 'name': 'Shahd', 'age': 20, 'department': 'AI', 'grade': 90}, {'id': 2, 'name': 'Hassan', 'age': 21, 'department': 'CS', 'grade': 85}, {'id': 3, 'name': 'Anas', 'age': 20, 'department': 'AI', 'grade': 95}, {'id': 4, 'name': 'Mariam', 'age': 22, 'department': 'CS', 'grade': 78}, {'id': 5, 'name': 'Omar', 'age': 21, 'department': 'IS', 'grade': 88}]


In [10]:
from pymongo import MongoClient

mongo = MongoClient(
    "mongodb://localhost:27017/",
    serverSelectionTimeoutMS=5000
)
db = mongo["students_db"]
collection = db["students"]

students_data = list(collection.find({}, {"_id": 0}))

print("Number of students:", len(students_data))
print(students_data)

Number of students: 5
[{'id': 1, 'name': 'Shahd', 'age': 20, 'department': 'AI', 'grade': 90}, {'id': 2, 'name': 'Hassan', 'age': 21, 'department': 'CS', 'grade': 85}, {'id': 3, 'name': 'Anas', 'age': 20, 'department': 'AI', 'grade': 95}, {'id': 4, 'name': 'Mariam', 'age': 22, 'department': 'CS', 'grade': 78}, {'id': 5, 'name': 'Omar', 'age': 21, 'department': 'IS', 'grade': 88}]


In [1]:
import os
print(os.listdir("/data"))

[]


In [35]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("StudentsBigDataProject")
    .master("spark://spark:7077")
    .config("spark.cores.max", "2")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/lakehouse/warehouse")
    .enableHiveSupport()
    .getOrCreate()
)

In [20]:
%pip install pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 247.1 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 212.0 kB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [24]:
from pymongo import MongoClient

client = MongoClient("mongodb://mongodb_db:27017")   # أو الـ connection string اللي معاكي
db = client["school_db"]

def collection_to_df(name):
    docs = list(db[name].find({}, {"_id": 0}))   # من غير عمود _id
    # نحول كل القيم لنصوص عشان تناسب Bronze (string)
    docs = [{k: str(v) for k, v in d.items()} for d in docs]
    return spark.createDataFrame(docs)

df_students    = collection_to_df("students")
df_attendances = collection_to_df("attendances")
df_grades      = collection_to_df("grades")

df_students.show()
df_attendances.show()
df_grades.show()

+---+----------+------------+----------+
|age|department|        name|student_id|
+---+----------+------------+----------+
| 20|        CS|   Ahmed Ali|         1|
| 21|        IT| Sara Hassan|         2|
| 20|        CS| Mona Khaled|         3|
| 22|        IS|Omar Youssef|         4|
| 21|        IT|   Nour Adel|         5|
+---+----------+------------+----------+

+----------+-------+----------+
|      date| status|student_id|
+----------+-------+----------+
|2026-10-01|present|         1|
|2026-10-02| absent|         1|
|2026-10-01|present|         2|
|2026-10-02|present|         2|
|2026-10-01| absent|         3|
|2026-10-02|present|         3|
|2026-10-01|present|         4|
|2026-10-01| absent|         5|
+----------+-------+----------+

+-------+-----+----------+
| course|grade|student_id|
+-------+-----+----------+
|   Math|   85|         1|
|Physics|   78|         1|
|   Math|   92|         2|
|   Math|   66|         3|
|Physics|   74|         4|
|   Math|   88|         5|
+-

Bronze Layer

In [25]:
bronze_path = "hdfs://namenode:9000/lakehouse/bronze"
df_students.write.mode("overwrite").parquet(f"{bronze_path}/students")
df_attendances.write.mode("overwrite").parquet(f"{bronze_path}/attendances")
df_grades.write.mode("overwrite").parquet(f"{bronze_path}/grades")

Silver Layer

In [33]:
from pyspark.sql import functions as F
silver_path = "hdfs://namenode:9000/lakehouse/silver"

s_students = (spark.read.parquet(f"{bronze_path}/students")
    .dropDuplicates(["student_id"])
    .withColumn("name", F.trim("name"))
    .withColumn("department", F.trim("department"))
    .withColumn("age", F.col("age").cast("int")))

s_attendances = (spark.read.parquet(f"{bronze_path}/attendances")
    .dropDuplicates(["student_id", "date"])
    .withColumn("date", F.to_date("date"))
    .withColumn("is_present", F.when(F.col("status") == "present", 1).otherwise(0)))

s_grades = (spark.read.parquet(f"{bronze_path}/grades")
    .withColumn("grade", F.col("grade").cast("int"))
    .filter((F.col("grade") >= 0) & (F.col("grade") <= 100)))

s_students.write.mode("overwrite").parquet(f"{silver_path}/students")
s_attendances.write.mode("overwrite").parquet(f"{silver_path}/attendances")
s_grades.write.mode("overwrite").parquet(f"{silver_path}/grades")

s_students.show()
print("Silver done")

+---+----------+------------+----------+
|age|department|        name|student_id|
+---+----------+------------+----------+
| 20|        CS|   Ahmed Ali|         1|
| 21|        IT| Sara Hassan|         2|
| 20|        CS| Mona Khaled|         3|
| 22|        IS|Omar Youssef|         4|
| 21|        IT|   Nour Adel|         5|
+---+----------+------------+----------+

Silver done


Gold Layer

In [34]:
gold_path = "hdfs://namenode:9000/lakehouse/gold"

avg_grades = s_grades.groupBy("student_id").agg(F.round(F.avg("grade"), 2).alias("avg_grade"))
att_rate = s_attendances.groupBy("student_id").agg(F.round(F.avg("is_present") * 100, 1).alias("attendance_pct"))

gold_students = (s_students
    .join(avg_grades, "student_id", "left")
    .join(att_rate, "student_id", "left")
    .select("student_id", "name", "department", "avg_grade", "attendance_pct"))

gold_dept = (gold_students.groupBy("department")
    .agg(F.round(F.avg("avg_grade"), 2).alias("avg_grade"),
         F.round(F.avg("attendance_pct"), 1).alias("avg_attendance_pct"),
         F.count("*").alias("students_count")))

gold_students.write.mode("overwrite").parquet(f"{gold_path}/student_summary")
gold_dept.write.mode("overwrite").parquet(f"{gold_path}/department_summary")

gold_students.show()
gold_dept.show()

+----------+------------+----------+---------+--------------+
|student_id|        name|department|avg_grade|attendance_pct|
+----------+------------+----------+---------+--------------+
|         1|   Ahmed Ali|        CS|     81.5|          50.0|
|         2| Sara Hassan|        IT|     92.0|         100.0|
|         3| Mona Khaled|        CS|     66.0|          50.0|
|         4|Omar Youssef|        IS|     74.0|         100.0|
|         5|   Nour Adel|        IT|     88.0|           0.0|
+----------+------------+----------+---------+--------------+

+----------+---------+------------------+--------------+
|department|avg_grade|avg_attendance_pct|students_count|
+----------+---------+------------------+--------------+
|        IT|     90.0|              50.0|             2|
|        CS|    73.75|              50.0|             2|
|        IS|     74.0|             100.0|             1|
+----------+---------+------------------+--------------+



In [36]:
spark.stop()